<a href="https://colab.research.google.com/github/vaasssuuu/SITL_Neurosymbolic_KG/blob/main/SITL_Neurosymbolic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =====================================================================
# 1. ENVIRONMENT SETUP & SERVER BOOT
# =====================================================================
print("1. Installing Python libraries...")
!pip install -q langchain langchain-community neo4j

import subprocess
import time
import urllib.request
from urllib.error import URLError
import os
import json
import re
import atexit
from neo4j import GraphDatabase
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import PromptTemplate

print("2. Installing zstd...")
!apt-get install zstd -y > /dev/null 2>&1

print("3. Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("4. Booting Ollama server in the background...")
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

try:
    urllib.request.urlopen('http://localhost:11434/')
    print("✅ Ollama server is running!")
except URLError as e:
    print(f"❌ Ollama server failed to start: {e}")

print("5. Pulling Llama 3 model...")
!ollama pull llama3 > /dev/null 2>&1
print("✅ Model ready!\n")

# =====================================================================
# 2. DATABASE & MODEL INITIALIZATION
# =====================================================================
from google.colab import userdata

# --- 1. Connect to your EMPWR -> Neo4j AuraDB SECURELY ---
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')

# Optional Safety Check
if not NEO4J_URI or not NEO4J_PASSWORD:
    raise ValueError("⚠️ Missing Credentials: Go to the Colab Secrets 🔑 menu and add your Neo4j keys!")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
atexit.register(driver.close)

# ---> THE FIX: Expanded num_predict to 512 so JSON never gets cut off <---
llm_chat = ChatOllama(model="llama3", temperature=0, num_predict=512, keep_alive=-1, timeout=300)
llm_extractor = ChatOllama(model="llama3", temperature=0, format="json", num_predict=512, keep_alive=-1, timeout=300)
llm_router = ChatOllama(model="llama3", temperature=0, format="json", num_predict=32, keep_alive=-1, timeout=300)

# =====================================================================
# 3. PROMPT ENGINEERING (WITH FEW-SHOT EXAMPLES)
# =====================================================================
router_prompt = PromptTemplate.from_template(
    """Analyze the user question and classify its intent into exactly ONE category:
    1. "FACT_CHECK": Specific factual questions about blockchain/crypto.
    2. "DISCOVERY": Broad questions SPECIFICALLY about blockchain/crypto.
    3. "OUT_OF_DOMAIN": Completely unrelated to blockchain or crypto.
    Return ONLY a JSON object with a single key "intent".
    Question: {question}"""
)

generate_prompt = PromptTemplate.from_template(
    """You are an expert in Blockchain technology. Answer the following question concisely based ONLY on verified knowledge: {question}

    If the question asks about something that did NOT happen, or asks about concepts that do not exist in your verified knowledge, simply state: "The verified knowledge graph does not contain information to support that claim." Then, optionally provide a relevant verified fact.

    {feedback}"""
)

# ---> THE FIX: Added Explicit Examples to force perfect triplet extraction <---
extract_prompt = PromptTemplate.from_template(
    """You are a strict factual extractor.
    User's Original Question: {question}
    Draft Response: {draft_response}

    CRITICAL RULES:
    1. If the text states "The verified knowledge graph does not contain...", you MUST return an empty array: []
    2. Extract ONLY the core factual triples answering the question. Ignore fluff.
    3. PRESERVE NEGATIONS: If the text says "X is not Y", the predicate MUST be "is not".

    EXAMPLES:
    Text: "No, Ethereum is not a centralized database." -> [{{"subject": "Ethereum", "predicate": "is not", "object": "centralized database"}}]
    Text: "Proof of Stake uses validators." -> [{{"subject": "Proof of Stake", "predicate": "uses", "object": "validators"}}]

    Format STRICTLY as a JSON list of dictionaries with keys 'subject', 'predicate', 'object'.
    """
)

# =====================================================================
# 4. KNOWLEDGE GRAPH LOGIC
# =====================================================================
def classify_intent(question):
    print(f"\n🚦 Routing Question: '{question}'")
    try:
        return json.loads((router_prompt | llm_router).invoke({"question": question}).content).get("intent", "DISCOVERY")
    except:
        return "DISCOVERY"

def check_entity_exists(tx, entity_name):
    query = "MATCH (n) WHERE toLower(n.name) CONTAINS toLower($name) OR toLower($name) CONTAINS toLower(n.name) RETURN count(n) > 0 AS exists"
    return tx.run(query, name=entity_name).single()["exists"]

def validate_triple_in_neo4j(tx, subject, predicate, obj):
    query = """MATCH (s)-[p]-(o)
               WHERE (toLower(s.name) CONTAINS toLower($subject) OR toLower($subject) CONTAINS toLower(s.name))
               AND (toLower(o.name) CONTAINS toLower($object) OR toLower($object) CONTAINS toLower(o.name))
               RETURN count(p) > 0 AS is_valid"""
    record = tx.run(query, subject=subject, object=obj).single()
    return record["is_valid"] if record else False

# =====================================================================
# 5. THE NEURO-SYMBOLIC PIPELINE
# =====================================================================
def neuro_symbolic_pipeline(question, max_iterations=3):
    print(f"\n{'='*50}\nProcessing Question: {question}\n{'='*50}")

    intent = classify_intent(question)
    print(f"🎯 Intent Identified: {intent}")
    if intent == "OUT_OF_DOMAIN":
        return "I am a specialized Blockchain AI. I cannot answer questions outside of this domain."

    feedback_text = ""

    for i in range(max_iterations):
        print(f"\n[Iteration {i+1}] 🧠 Generating Neural Output...")
        draft_response = (generate_prompt | llm_chat).invoke({"question": question, "feedback": feedback_text}).content
        print(f"Draft Response:\n{draft_response}\n")

        if "does not contain information" in draft_response.lower() or "does not contain evidence" in draft_response.lower():
            print("🛡️ Escape Hatch Triggered.")
            return draft_response

        print(f"[Iteration {i+1}] 🔍 Extracting Triples for Validation...")
        triples_json_str = (extract_prompt | llm_extractor).invoke({"question": question, "draft_response": draft_response}).content

        try:
            clean_json = triples_json_str.replace('```json', '').replace('```', '').strip()
            parsed_data = json.loads(clean_json)
            if isinstance(parsed_data, dict):
                triples = [parsed_data] if 'subject' in parsed_data else next((v for v in parsed_data.values() if isinstance(v, list)), [])
            elif isinstance(parsed_data, list):
                triples = parsed_data
            else:
                triples = []
        except json.JSONDecodeError:
            print("⚠️ Failed to parse JSON from LLM. Retrying iteration...")
            continue

        print(f"Extracted Triples: {json.dumps(triples, indent=2)}\n")

        print(f"[Iteration {i+1}] ⚖️ Validating against Knowledge Graph...")
        all_valid, invalid_triples, valid_triples_list = True, [], []

        with driver.session() as session:
            for triple in triples:
                subj = re.sub(r'^(a|an|the)\s+', '', str(triple.get('subject', '')), flags=re.IGNORECASE).strip()
                pred = str(triple.get('predicate', '')).strip()
                obj = re.sub(r'^(a|an|the)\s+', '', str(triple.get('object', '')), flags=re.IGNORECASE).strip()

                if not session.execute_read(check_entity_exists, subj) or not session.execute_read(check_entity_exists, obj):
                    all_valid = False
                    invalid_triples.append(triple)
                    print(f"  ❌ Hallucination Detected: Entities '{subj}' or '{obj}' do not exist in KG.")
                    continue

                if not session.execute_read(validate_triple_in_neo4j, subj, pred, obj):
                    all_valid = False
                    invalid_triples.append(triple)
                    print(f"  ❌ Hallucination Detected: ({subj})-[{pred}]->({obj}) is NOT in the KG boundary.")
                else:
                    valid_triples_list.append(triple)
                    print(f"  ✅ Validated: ({subj})-[{pred}]->({obj})")

        if all_valid:
            print("\n✔️ FINAL VALIDATED RESPONSE:")
            return draft_response
        else:
            print("\n🔄 Out-of-bounds concepts found. Re-prompting the LLM...")
            failed_claims_str = ", ".join([f"({t.get('subject')}) -> ({t.get('object')})" for t in invalid_triples])
            valid_claims_str = ", ".join([f"({t.get('subject')}) -> ({t.get('object')})" for t in valid_triples_list])

            if len(valid_triples_list) > 0:
                feedback_text = f"IMPORTANT INSTRUCTION: Your previous attempt was partially correct! These claims were VALIDATED: {valid_claims_str}. However, you added extra context that was REJECTED: {failed_claims_str}. Rewrite your response to answer using ONLY the validated claims."
            else:
                feedback_text = f"CRITICAL SYSTEM OVERRIDE: Your previous attempt included these claims which were completely REJECTED: {failed_claims_str}. You are experiencing a Knowledge Conflict. You MUST output this EXACT sentence and nothing else: 'The verified knowledge graph does not contain information to support that claim.'"

    return f"Error: Unable to generate a fully validated response within {max_iterations} iterations."

def extract_and_validate(text, question):
    triples_json_str = (extract_prompt | llm_extractor).invoke({"question": question, "draft_response": text}).content
    try:
        clean_json = triples_json_str.replace('```json', '').replace('```', '').strip()
        parsed_data = json.loads(clean_json)
        triples = [parsed_data] if isinstance(parsed_data, dict) and 'subject' in parsed_data else (parsed_data if isinstance(parsed_data, list) else [])
    except json.JSONDecodeError:
        triples = []

    total_triples, hallucinated_triples = len(triples), 0
    with driver.session() as session:
        for t in triples:
            subj = re.sub(r'^(a|an|the)\s+', '', str(t.get('subject', '')), flags=re.IGNORECASE).strip()
            pred = str(t.get('predicate', '')).strip()
            obj = re.sub(r'^(a|an|the)\s+', '', str(t.get('object', '')), flags=re.IGNORECASE).strip()

            if not session.execute_read(check_entity_exists, subj) or not session.execute_read(check_entity_exists, obj) or not session.execute_read(validate_triple_in_neo4j, subj, pred, obj):
                hallucinated_triples += 1

    return total_triples, hallucinated_triples

# =====================================================================
# 6. EVALUATION EXECUTION LOOP
# =====================================================================
benchmark_file = 'benchmark.json'
if not os.path.exists(benchmark_file): raise FileNotFoundError(f"⚠️ Cannot find '{benchmark_file}'. Please upload your JSON file!")

with open(benchmark_file, 'r') as f: benchmark_data = json.load(f)

progress_file = 'semantic_drift_progress.json'
results_memory = json.load(open(progress_file, 'r')) if os.path.exists(progress_file) else []
i = len(results_memory)

print(f"\n🔬 STARTING SEMANTIC DRIFT EVALUATION ({len(benchmark_data)} Questions)...\n")

while i < len(benchmark_data):
    item = benchmark_data[i]
    q, category = item['question'], item.get('category', 'Unknown')
    print(f"\n--- [{i+1}/{len(benchmark_data)}] Category: {category} ---")

    try:
        time.sleep(1) # Let the T4 breathe
        draft_1 = (generate_prompt | llm_chat).invoke({"question": q, "feedback": ""}).content
        base_tot, base_hal = extract_and_validate(draft_1, q)

        final_response = neuro_symbolic_pipeline(q)
        if "cannot answer questions outside of this domain" in final_response or "does not contain information" in final_response.lower():
            final_tot, final_hal = 0, 0
        else:
            final_tot, final_hal = extract_and_validate(final_response, q)

        results_memory.append({"question": q, "base_tot": base_tot, "base_hal": base_hal, "final_tot": final_tot, "final_hal": final_hal})
        with open(progress_file, 'w') as f: json.dump(results_memory, f, indent=4)

        print(f"✅ Saved! Baseline Hallucinations: {base_hal}/{base_tot} | Final Hallucinations: {final_hal}/{final_tot}")
        i += 1

    except Exception as e:
        print(f"\n⚠️ OLLAMA CRASH DETECTED: {e}")
        print("🛠️ AUTO-HEALING: Rebooting the server...")
        os.system("pkill ollama")
        time.sleep(3)
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        time.sleep(10)
        print(f"✅ Server rebooted! Retrying Question {i+1} now...\n")

# =====================================================================
# 7. FINAL MATH & METRICS
# =====================================================================
if len(results_memory) == len(benchmark_data):
    tb_triples = sum(r['base_tot'] for r in results_memory)
    tb_hals = sum(r['base_hal'] for r in results_memory)
    tf_triples = sum(r['final_tot'] for r in results_memory)
    tf_hals = sum(r['final_hal'] for r in results_memory)

    H_base = (tb_hals / tb_triples) if tb_triples > 0 else 0
    H_final = (tf_hals / tf_triples) if tf_triples > 0 else 0
    drift_reduction = ((H_base - H_final) / H_base) * 100 if H_base > 0 else 0

    print("\n" + "="*60)
    print("📊 FINAL IEEE EVALUATION METRICS: SEMANTIC DRIFT 📊")
    print("="*60)
    print(f"Total Baseline Triples Extracted: {tb_triples}")
    print(f"Baseline Hallucinations (Raw LLM): {tb_hals}")
    print(f"-> Baseline Hallucination Rate (H_base): {H_base * 100:.2f}%\n")
    print(f"Total Final Triples Extracted: {tf_triples}")
    print(f"Final Hallucinations (Pipeline): {tf_hals}")
    print(f"-> Final Hallucination Rate (H_final): {H_final * 100:.2f}%\n")
    print(f"🚀 SEMANTIC DRIFT REDUCTION: {drift_reduction:.2f}%")
    print("="*60)

1. Installing Python libraries...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
2. Installing zstd...
3. Installing Ollama...
4. Booting Ollama server in the background...
✅ Ollama server is running!
5. Pulling Llama 3 model...
✅ Model ready!


🔬 STARTING SEMANTIC DRIFT EVALUATION (50 Questions)...


--- [1/50] Category: Direct Facts ---


/tmp/ipykernel_6988/2169679776.py:50: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm_chat = ChatOllama(model="llama3", temperature=0, num_predict=512, keep_alive=-1, timeout=300)



Processing Question: Does blockchain require consensus?

🚦 Routing Question: 'Does blockchain require consensus?'
🎯 Intent Identified: FACT_CHECK

[Iteration 1] 🧠 Generating Neural Output...
Draft Response:
Yes, blockchain technology typically requires consensus among nodes or participants to validate and record transactions on the distributed ledger. This ensures the integrity and security of the network by preventing any single entity from manipulating the data.

[Iteration 1] 🔍 Extracting Triples for Validation...
Extracted Triples: [
  {
    "subject": "blockchain technology",
    "predicate": "requires",
    "object": "consensus"
  }
]

[Iteration 1] ⚖️ Validating against Knowledge Graph...
  ✅ Validated: (blockchain technology)-[requires]->(consensus)

✔️ FINAL VALIDATED RESPONSE:
✅ Saved! Baseline Hallucinations: 0/1 | Final Hallucinations: 0/1

--- [2/50] Category: Direct Facts ---

Processing Question: Is SHA-256 an example of cryptographic hash functions?

🚦 Routing Question